In [1]:
import os
import sys

try:
    # Для Jupyter Notebook
    notebook_path = os.path.join(os.getcwd(), sys.argv[0])
except:
    # Для обычного Python скрипта
    notebook_path = os.path.abspath(__file__)

notebook_dir = os.path.dirname(notebook_path)
print("Директория файла:", notebook_dir)

Директория файла: C:\Users\looki\AppData\Roaming\Python\Python38\site-packages


In [ ]:
import os
import cv2
import random
import numpy as np
from natsort import natsorted
import math

class Image_generator:
  def __init__(self, data_dir, slices=4):
    self.__data_dir = data_dir
    self.__nof_slices = slices

  def slice(self):
    self.__res_slice = {}
    data = dict()
    for root, dirs, files in os.walk(self.__data_dir):
      if len(files) != 0:
        lable = root.replace('\\', '/')
        data[lable] = natsorted(files)

    self.__lock_size = False
    for key in data.keys():
      lable = key[key.rfind('/') + 1:]
      self.__res_slice[lable] = []
      for img_name in data[key]:
        self.__slice_image(lable, key, img_name)

  def __slice_image(self, lable, src_path, img_name):
    os.chdir(src_path)
    src_img = cv2.imread(img_name)
  
    if not self.__lock_size:
      height, width = src_img.shape[:2]
      self.__height = height
      self.__width = width
      self.__lock_size = True
    else:
      src_img = cv2.resize(src_img, dsize=(self.__width, self.__height), 
                           interpolation=cv2.INTER_CUBIC)
    
    tile_height = self.__height // (self.__nof_slices // 2)
    tile_width = self.__width // (self.__nof_slices // 2)

    slice_cntr = 0
    for y in range(0, self.__height, tile_height):
      for x in range(0, self.__width, tile_width):
        y_end = min(y + tile_height, self.__height)
        x_end = min(x + tile_width, self.__width)

        tile = src_img[y:y_end, x:x_end]
        self.__res_slice[lable].append(tile)
        slice_cntr += 1

    #print(self.__res_data)

  def glue(self, nof_res_imgs=1, nof_fragments=4):
    self.__res_glue = {
      list(self.__res_slice.keys())[0]: [],
      list(self.__res_slice.keys())[1]: []
      }
    gen_idxs = [[random.randint(0, 
                 len(self.__res_slice[list(self.__res_slice.keys())[0]]) - 1)
                 for _ in range(nof_fragments)] for _ in range(nof_res_imgs)]

    for idxs in gen_idxs:
      self.__glue_fragments(idxs)
      
  def __glue_fragments(self, idxs):
    img_frags = np.array([self.__res_slice[list(self.__res_slice.keys())[0]][idx] for idx in idxs])
    mask_frags = np.array([self.__res_slice[list(self.__res_slice.keys())[1]][idx] for idx in idxs])
    img_frags = img_frags.reshape(int(np.sqrt(img_frags.shape[0])), int(np.sqrt(img_frags.shape[0])), *img_frags.shape[1:])
    mask_frags = mask_frags.reshape(int(np.sqrt(mask_frags.shape[0])), int(np.sqrt(mask_frags.shape[0])), *mask_frags.shape[1:])

    self.__res_glue[list(self.__res_glue.keys())[0]].append(cv2.resize(cv2.vconcat([cv2.hconcat(list_h) for list_h in img_frags]), dsize=(self.__width, self.__height), 
                                                                       interpolation=cv2.INTER_CUBIC))
    self.__res_glue[list(self.__res_glue.keys())[1]].append(cv2.resize(cv2.vconcat([cv2.hconcat(list_h) for list_h in mask_frags]), dsize=(self.__width, self.__height), 
                                                                       interpolation=cv2.INTER_CUBIC))
    
    '''self.__res_glue[list(self.__res_glue.keys())[0]].append(cv2.vconcat([cv2.hconcat(list_h) for list_h in img_frags]))
    self.__res_glue[list(self.__res_glue.keys())[1]].append(cv2.vconcat([cv2.hconcat(list_h) for list_h in mask_frags]))'''

  def __getitem__(self,key):
    return {"img": self.__res_glue[list(self.__res_glue.keys())[0]][key],
            "mask": self.__res_glue[list(self.__res_glue.keys())[1]][key]}

    

In [ ]:
dir_ = "path".replace('\\', '/')
obj = Image_generator(dir_)
obj.slice()
obj.glue(1, 16)
print(obj[0])

(2400, 3200, 3)
(2400, 3200, 3)
{'img': array([[[217, 221, 215],
        [215, 221, 216],
        [223, 229, 224],
        ...,
        [236, 163, 195],
        [238, 168, 198],
        [241, 171, 200]],

       [[221, 224, 222],
        [221, 224, 222],
        [222, 224, 224],
        ...,
        [230, 159, 189],
        [236, 165, 193],
        [236, 167, 193]],

       [[226, 229, 227],
        [226, 229, 227],
        [224, 226, 226],
        ...,
        [220, 151, 179],
        [230, 160, 186],
        [233, 163, 188]],

       ...,

       [[245, 240, 242],
        [245, 240, 242],
        [242, 237, 239],
        ...,
        [149, 104,  97],
        [155, 109, 101],
        [155, 111, 102]],

       [[245, 240, 242],
        [245, 240, 242],
        [242, 237, 239],
        ...,
        [146, 103,  94],
        [143, 103,  92],
        [141, 103,  91]],

       [[245, 240, 242],
        [245, 240, 242],
        [242, 237, 239],
        ...,
        [138,  99,  88],
        [

True